# Leakage-audited load and price forecasting — full analysis

**Paper:** *Leakage-Audited Benchmarking of Machine Learning, Deep Learning and Time-Series Foundation Models for Day-Ahead Load and Electricity Price Forecasting* (Energy Conversion and Economics, special issue "AI and Large Language Models: Transforming Energy & Power Systems").

Run the cells **top to bottom**. Each STEP prints or plots its result underneath. Every model run is **saved to disk the first time** (folder `OUT/cache`), so if you stop and run again, finished runs load instantly instead of retraining.

| Steps | Content |
|---|---|
| 1 | Settings: where your data files are |
| 2–5 | Load ISO-NE (main) and Tetouan (secondary), data figures |
| 6–7 | Information sets, purged split, metrics, significance test |
| 8–14 | ISO-NE load: benchmarks, tree ensembles, deep models, Chronos, significance, summary |
| 15–18 | Tetouan load: same models, plus rolling-origin evaluation |
| 19–23 | Day-ahead price: ISO-NE and EPF benchmark markets |
| 24 | Final tables for the paper |

In [ ]:
# ===================================================================
# STEP 1 — SETTINGS (edit the paths if your files are elsewhere)
# ===================================================================
import os, time, json, copy, random, warnings, pickle, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

HOME = Path.home()
DATA = HOME / "Downloads" / "open-forecasting-framework-v2" / "data" / "raw"
ISONE_FILES = [DATA / "isone" / f"{y}_smd_hourly.xlsx" for y in (2023, 2024, 2025)]
TETOUAN_CSV = DATA / "power_consumption.csv"
EPF_DIR = HOME / "Downloads" / "epf_data"               # official EPF files are downloaded here (STEP 23)
OUT = HOME / "Downloads" / "forecast_results"           # every table, figure and cached run goes here

RUN_DEEP = True          # LSTM / Transformer / PatchTST (slow on a laptop: hours)
RUN_FOUNDATION = True    # Chronos-Bolt and Chronos-2 (needs internet the first time)
RUN_EPF = True           # EPF benchmark markets for price (30-60 min per market)
DEVICE = "mps"           # "mps" on Apple silicon, "cuda" on NVIDIA, "cpu" otherwise
SEED = 42
H = 24                   # forecast horizon (hours)

(OUT / "cache").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
for f in ISONE_FILES + [TETOUAN_CSV]:
    print("OK     " if f.exists() else "MISSING", f)

def cached(name, compute):
    """Run compute() once and save the result; later runs load it from disk."""
    path = OUT / "cache" / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as fh:
            print(f"[loaded from cache] {name}")
            return pickle.load(fh)
    start = time.time()
    result = compute()
    with open(path, "wb") as fh:
        pickle.dump(result, fh)
    print(f"[computed in {(time.time() - start) / 60:.1f} min] {name}")
    return result

In [ ]:
# ===================================================================
# STEP 2 — LOAD ISO NEW ENGLAND ZONAL LOAD AND WEATHER (main dataset)
# Declared daylight-saving policy:
#  * Hr_End is hour-ending local time. 2024-2025 files follow the true clock
#    (23-h day in March, 25-h day in November with "02X").
#  * 2023 uses a legacy 24-rows-per-day convention: the synthetic spring HE02 is
#    dropped; the averaged autumn HE02 is kept and the second hour interpolated.
#  * Timestamps: hour-beginning on a fixed UTC-05:00 clock (continuous index).
# ===================================================================
ZONES = ["ME", "NH", "VT", "CT", "RI", "SEMA", "WCMA", "NEMA"]
LOCAL_TZ, FIXED_OFFSET = "America/New_York", pd.Timedelta(hours=-5)

def load_isone_zone(path, zone):
    frame = pd.read_excel(path, sheet_name=zone, usecols=["Date", "Hr_End", "RT_Demand", "Dry_Bulb", "Dew_Point"])
    frame["h"] = frame["Hr_End"].astype(str).str.strip()
    frame["Date"] = pd.to_datetime(frame["Date"])
    counts = {"spring_dropped": 0, "autumn_interpolated": 0}
    days = []
    for day, g in frame.groupby("Date", sort=True):
        g = g.copy()
        start = pd.Timestamp(day).tz_localize(LOCAL_TZ)
        end = pd.Timestamp(day + pd.Timedelta(days=1)).tz_localize(LOCAL_TZ)
        real = int((end - start) / pd.Timedelta(hours=1))
        if len(g) == 24 and real == 23:
            g = g[g["h"].str.lstrip("0") != "2"]; counts["spring_dropped"] += 1
        if len(g) == 24 and real == 25:
            pos = g.index[g["h"].str.lstrip("0") == "2"][0]
            gap = g.loc[[pos]].copy(); gap[["RT_Demand", "Dry_Bulb", "Dew_Point"]] = np.nan
            g = pd.concat([g.loc[:pos], gap, g.loc[pos + 1:]]); counts["autumn_interpolated"] += 1
        assert len(g) == real, f"{path.name} {zone} {day.date()}: {len(g)} rows for {real} hours"
        utc = start.tz_convert("UTC") + pd.to_timedelta(np.arange(real), unit="h")
        g.index = (utc + FIXED_OFFSET).tz_localize(None)
        days.append(g[["RT_Demand", "Dry_Bulb", "Dew_Point"]])
    return pd.concat(days), counts

def load_isone_load():
    columns, audit = {}, {}
    for zone in ZONES:
        parts = []
        for path in ISONE_FILES:
            data, c = load_isone_zone(path, zone); parts.append(data)
            for k, v in c.items(): audit[k] = audit.get(k, 0) + v
        z = pd.concat(parts)
        columns[zone] = z["RT_Demand"]
        columns[f"{zone}_Dry_Bulb"] = z["Dry_Bulb"]; columns[f"{zone}_Dew_Point"] = z["Dew_Point"]
    hourly = pd.DataFrame(columns)
    hourly = hourly.reindex(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h"))
    audit["cells_interpolated"] = int(hourly.isna().sum().sum())
    return hourly.interpolate(limit=2), audit

isone, isone_audit = cached("data_isone_load", load_isone_load)
ISONE_WEATHER = [f"{z}_{v}" for z in ZONES for v in ("Dry_Bulb", "Dew_Point")]
print("Shape:", isone.shape, "| from", isone.index.min(), "to", isone.index.max())
print("Daylight-saving audit (all zones):", isone_audit)
display(isone[ZONES].describe().round(1))

In [ ]:
# ===================================================================
# STEP 3 — LOAD ISO-NE SYSTEM DAY-AHEAD PRICE
# Price days use 24 local wall-clock hours (EPF convention): the missing spring
# hour is interpolated, the repeated autumn hour (02 and 02X) is averaged.
# DA_Demand is NOT loaded: it is cleared by the same auction as the price.
# ===================================================================
def load_isone_price():
    cols = ["DA_LMP", "RT_Demand", "Dry_Bulb", "Dew_Point"]
    f = pd.concat([pd.read_excel(p, sheet_name="ISO NE CA", usecols=["Date", "Hr_End", *cols]) for p in ISONE_FILES], ignore_index=True)
    he = pd.to_numeric(f["Hr_End"].astype(str).str.strip().str.replace("X", ""))
    f["ts"] = pd.to_datetime(f["Date"]) + pd.to_timedelta(he - 1, unit="h")
    hourly = f.groupby("ts")[cols].mean()
    hourly = hourly.reindex(pd.date_range(hourly.index.min(), hourly.index.max(), freq="h")).interpolate(limit=2)
    return hourly.rename(columns={"DA_LMP": "Price"})

isone_price = cached("data_isone_price", load_isone_price)
p = isone_price["Price"]
print("Hourly DA LMP ($/MWh): mean %.2f | median %.2f | 99th pct %.2f | max %.2f | hours > $100: %.1f%%"
      % (p.mean(), p.median(), p.quantile(0.99), p.max(), 100 * (p > 100).mean()))
display(isone_price.head())

In [ ]:
# ===================================================================
# STEP 4 — LOAD TETOUAN (secondary dataset)
# 10-minute data -> hourly means; only complete hours are allowed.
# The source does not state a unit: values are kept "as published".
# ===================================================================
TET_ZONES = ["Zone 1", "Zone 2"]
TET_WEATHER = ["Temperature", "Humidity", "Wind Speed", "general diffuse flows", "diffuse flows"]

def load_tetouan():
    raw = pd.read_csv(TETOUAN_CSV)
    raw.columns = [" ".join(str(c).split()).replace(" Power Consumption", "") for c in raw.columns]
    text = raw["DateTime"].astype(str).str.strip()
    idx = pd.to_datetime(text, format="%m/%d/%Y %H:%M", errors="coerce")
    idx = idx.fillna(pd.to_datetime(text, format="%m-%d-%Y %H:%M", errors="coerce"))
    assert idx.notna().all() and idx.is_monotonic_increasing and not idx.duplicated().any()
    data = raw[TET_ZONES + TET_WEATHER].set_index(pd.DatetimeIndex(idx))
    counts = data.resample("h").count()
    assert (counts == 6).all().all(), "incomplete hours found"
    return data.resample("h").mean()

tetouan = cached("data_tetouan", load_tetouan)
print("Shape:", tetouan.shape, "| from", tetouan.index.min(), "to", tetouan.index.max())
display(tetouan.describe().round(2))

In [ ]:
# ===================================================================
# STEP 5 — DATA FIGURES
# ===================================================================
daily = isone_price.resample("D").agg({"RT_Demand": "mean", "Price": "mean", "Dry_Bulb": "mean"})
fig, (a, b) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
a.plot(daily.index, daily["RT_Demand"] / 1000, color="#2a78d6"); a.set_ylabel("System load (GW)")
b.plot(daily.index, daily["Price"], color="#eb6834"); b.set_ylabel("DA LMP ($/MWh)")
for ax in (a, b): ax.axvspan(pd.Timestamp("2025-01-01"), daily.index.max(), color="#e4e3df", zorder=0)
a.set_title("ISO-NE daily means (shaded: test year 2025)"); plt.tight_layout()
plt.savefig(OUT / "figures" / "isone_overview.png", dpi=300); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, year in enumerate(sorted(daily.index.year.unique())):
    d = daily[daily.index.year == year]
    ax[0].scatter(d["Dry_Bulb"], d["RT_Demand"] / 1000, s=8, alpha=0.7, label=str(year))
ax[0].set_xlabel("Daily mean temperature (°F)"); ax[0].set_ylabel("Daily mean load (GW)"); ax[0].legend()
ax[1].hist(isone_price["Price"], bins=120, log=True, color="#eb6834"); ax[1].set_xlabel("Hourly DA LMP ($/MWh)"); ax[1].set_ylabel("Hours (log)")
plt.tight_layout(); plt.savefig(OUT / "figures" / "isone_load_temp_price_tail.png", dpi=300); plt.show()

week = tetouan.loc["2017-10-09":"2017-10-15 23:00", TET_ZONES] / 1000
week.plot(figsize=(10, 3), title="Tetouan, a regular week (dataset units x1000)"); plt.tight_layout(); plt.show()

In [ ]:
# ===================================================================
# STEP 6 — INFORMATION SETS AND HORIZON-PURGED SPLIT
#  A  weather_time                : weather at t+H (declared proxy) + calendar of t+H
#  B  weather_time_history        : A + load lags at t-1,2,3,6,12,24 h + 3 h / 24 h rolling means
#  B+ weather_time_history_weekly : B + load at t-144 h and t-168 h
# ===================================================================
MODES = ["weather_time", "weather_time_history", "weather_time_history_weekly"]
MODE_NAME = {"weather_time": "A", "weather_time_history": "B", "weather_time_history_weekly": "B+"}
LAGS, ROLLS, WEEKLY = [1, 2, 3, 6, 12, 24], [3, 24], [144, 168]

def calendar(index):
    return pd.DataFrame({"hour_sin": np.sin(2 * np.pi * index.hour / 24), "hour_cos": np.cos(2 * np.pi * index.hour / 24),
                         "dow_sin": np.sin(2 * np.pi * index.dayofweek / 7), "dow_cos": np.cos(2 * np.pi * index.dayofweek / 7),
                         "is_weekend": (index.dayofweek >= 5).astype(float)}, index=index)

def tabular(hourly, targets, weather, mode):
    x = hourly[weather].shift(-H).add_prefix("target_weather_")
    cal = calendar(hourly.index + pd.Timedelta(hours=H)); cal.index = hourly.index
    x = pd.concat([x, cal], axis=1)
    if mode != "weather_time":
        for z in targets:
            for lag in LAGS: x[f"{z}_lag_{lag}h"] = hourly[z].shift(lag)
            for w in ROLLS: x[f"{z}_roll_{w}h"] = hourly[z].rolling(w).mean()
            if mode == "weather_time_history_weekly":
                for lag in WEEKLY: x[f"{z}_weekly_{lag}h"] = hourly[z].shift(lag)
    y = hourly[targets].shift(-H)
    ok = x.notna().all(axis=1) & y.notna().all(axis=1)
    return x.loc[ok], y.loc[ok]

def purged_split(index, train_frac, val_frac):
    n = len(index); tr = int(n * train_frac); ve = tr + int(n * val_frac)
    return {"train": index[:tr], "validation": index[tr + H:ve], "test": index[ve + H:]}

def prepare(hourly, targets, weather, train_frac, val_frac):
    frames = {m: tabular(hourly, targets, weather, m) for m in MODES}
    common = frames[MODES[0]][0].index
    for m in MODES[1:]: common = common.intersection(frames[m][0].index)
    frames = {m: (x.loc[common], y.loc[common]) for m, (x, y) in frames.items()}
    return frames, purged_split(common, train_frac, val_frac)

iso_frames, iso_split = prepare(isone, ZONES, ISONE_WEATHER, 0.5, 0.1667)
tet_frames, tet_split = prepare(tetouan, TET_ZONES, TET_WEATHER, 0.7, 0.15)
for name, sp in [("ISO-NE", iso_split), ("Tetouan", tet_split)]:
    print(f"\n{name} split (forecast origins, {H}-h purge between partitions):")
    display(pd.DataFrame({k: [len(v), v.min(), v.max()] for k, v in sp.items()}, index=["origins", "first", "last"]).T)
    assert sp["train"].max() + pd.Timedelta(hours=H) < sp["validation"].min()
    assert sp["validation"].max() + pd.Timedelta(hours=H) < sp["test"].min()
print("Purge check passed. Features per mode (ISO-NE):", {MODE_NAME[m]: f.shape[1] for m, (f, _) in iso_frames.items()})

In [ ]:
# ===================================================================
# STEP 7 — METRICS AND DIEBOLD-MARIANO TEST
# ===================================================================
def scores(actual, pred, zones, model, mode):
    rows = []
    for j, z in enumerate(zones + ["aggregate"]):
        a = actual if z == "aggregate" else actual[:, [j]]; p = pred if z == "aggregate" else pred[:, [j]]
        e = a - p
        rows.append({"model": model, "set": MODE_NAME.get(mode, mode), "scope": z,
                     "RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(),
                     "nRMSE_%": 100 * np.sqrt((e ** 2).mean()) / np.abs(a).mean(),
                     "MAPE_%": 100 * np.mean(np.abs(e) / np.abs(a))})
    return pd.DataFrame(rows)

def dm_test(e_model, e_bench, h=H):
    """DM on squared errors, Newey-West variance with h-1 lags, HLN correction. stat < 0: model better."""
    d = e_model ** 2 - e_bench ** 2; n = len(d); dc = d - d.mean()
    gamma = [np.dot(dc[k:], dc[:n - k]) / n for k in range(h)]
    var = max(gamma[0] + 2 * sum((1 - k / h) * gamma[k] for k in range(1, h)), 1e-12)
    stat = d.mean() / np.sqrt(var / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return stat, 2 * stats.t.sf(abs(stat), df=n - 1)

RESULTS = {}   # (dataset, model, mode) -> {"pred": array, "actual": array}
def record(dataset, model, mode, actual, pred, zones):
    RESULTS[(dataset, model, mode)] = {"actual": actual, "pred": pred}
    return scores(actual, pred, zones, model, mode)

def show(table, title):
    agg = table[table["scope"] == "aggregate"].drop(columns="scope")
    print(title); display(agg.round(2).reset_index(drop=True))

In [ ]:
# ===================================================================
# STEP 8 — ISO-NE BENCHMARKS: persistence and weekly seasonal naive
# ===================================================================
def benchmarks(hourly, targets, split, dataset):
    test = split["test"]
    actual = hourly[targets].shift(-H).loc[test].to_numpy()
    out = []
    for model, offset in [("persistence", 0), ("seasonal_naive", 168 - H)]:
        pred = hourly.loc[test - pd.Timedelta(hours=offset), targets].to_numpy()
        out.append(record(dataset, model, "weather_time_history", actual, pred, targets))
    return pd.concat(out)

iso_bench = benchmarks(isone, ZONES, iso_split, "isone")
show(iso_bench, "ISO-NE benchmarks, test year 2025 (MW):")

In [ ]:
# ===================================================================
# STEP 9 — ISO-NE TREE ENSEMBLES (one model per zone, all three information sets)
# Random forest is the slowest (can take ~1 h); each run is cached.
# ===================================================================
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.base import clone

TREE_MODELS = {
    "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=200, max_depth=10, learning_rate=0.05, random_state=SEED),
    "extra_trees": ExtraTreesRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=SEED),
    "random_forest": RandomForestRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=SEED),
}

def fit_trees(frames, split, model_name, mode):
    x, y = frames[mode]
    xtr, ytr, xte = x.loc[split["train"]], y.loc[split["train"]], x.loc[split["test"]]
    pred = np.column_stack([clone(TREE_MODELS[model_name]).fit(xtr, ytr.iloc[:, j]).predict(xte) for j in range(y.shape[1])])
    return y.loc[split["test"]].to_numpy(), pred

iso_tables = [iso_bench]
for model_name in TREE_MODELS:
    for mode in MODES:
        actual, pred = cached(f"isone_{model_name}_{mode}", lambda: fit_trees(iso_frames, iso_split, model_name, mode))
        iso_tables.append(record("isone", model_name, mode, actual, pred, ZONES))
    show(pd.concat([t for t in iso_tables if (t["model"] == model_name).any()]), f"\nISO-NE {model_name} (MW):")

In [ ]:
# ===================================================================
# STEP 10 — DEEP SEQUENCE MODELS (LSTM, cross-attention Transformer, PatchTST)
# 24-h lookback of the same information set, all zones jointly; AdamW, MSE,
# early stopping on the validation split. Set RUN_DEEP = False to skip.
# ===================================================================
if RUN_DEEP:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    def sequences(hourly, targets, weather, mode, split, lookback=24):
        exo = pd.concat([hourly[weather].shift(-H), calendar(hourly.index + pd.Timedelta(hours=H)).set_index(hourly.index)], axis=1).to_numpy(np.float32)
        tv = hourly[targets].to_numpy(np.float32)
        weekly = [hourly[targets].shift(l).to_numpy(np.float32) for l in WEEKLY]
        pos = {t: i for i, t in enumerate(hourly.index)}
        parts = {}
        for name, idx in split.items():
            xs, ys = [], []
            for t in idx:
                end = pos[t]; s = end - lookback + 1
                per_zone = []
                for j in range(len(targets)):
                    block = [exo[s:end + 1]]
                    if mode != "weather_time":
                        block.append(tv[s:end + 1, j:j + 1])
                        if mode == "weather_time_history_weekly":
                            block += [w[s:end + 1, j:j + 1] for w in weekly]
                    per_zone.append(np.concatenate(block, axis=1))
                xs.append(np.stack(per_zone)); ys.append(tv[end + H])
            parts[name] = (np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32))
        return parts

    class LSTMNet(nn.Module):
        def __init__(s, f, z, hidden=128, layers=2, dropout=0.1):
            super().__init__()
            s.lstm = nn.LSTM(f * z, hidden, layers, batch_first=True, dropout=dropout, bidirectional=True)
            s.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, z))
        def forward(s, x):
            out, _ = s.lstm(x.permute(0, 2, 1, 3).flatten(2)); return s.head(out[:, -1])

    class CrossBlock(nn.Module):
        def __init__(s, d, heads, dropout):
            super().__init__()
            s.att = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
            s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
            s.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))
        def forward(s, q, c):
            a, _ = s.att(q, c, c, need_weights=False); x = s.n1(q + a); return s.n2(x + s.ff(x))

    class CrossAttentionNet(nn.Module):
        """Each zone keeps its own stream and attends to the mean of the other zones."""
        def __init__(s, f, z, d=64, heads=4, layers=2, dropout=0.1):
            super().__init__(); s.z = z
            s.proj = nn.Linear(f, d)
            layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True, norm_first=True)
            s.self_enc = nn.ModuleList([nn.TransformerEncoder(copy.deepcopy(layer), 1) for _ in range(z)])
            s.blocks = nn.ModuleList([CrossBlock(d, heads, dropout) for _ in range(layers)])
            s.head = nn.Sequential(nn.LayerNorm(d * z), nn.Linear(d * z, z))
        def forward(s, x):
            st = [s.self_enc[i](s.proj(x[:, i])) for i in range(s.z)]
            for b in s.blocks:
                st = [b(st[i], torch.stack([o for k, o in enumerate(st) if k != i]).mean(0)) for i in range(s.z)]
            return s.head(torch.cat([t.mean(1) for t in st], 1))

    class PatchTSTNet(nn.Module):
        def __init__(s, f, z, d=64, heads=4, layers=2, patch=4, stride=2, dropout=0.1):
            super().__init__(); s.patch, s.stride = patch, stride
            s.proj = nn.Linear(patch * f, d)
            s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True, norm_first=True), layers)
            s.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        def forward(s, x):
            b, z, l, f = x.shape
            p = x.unfold(2, s.patch, s.stride).permute(0, 1, 2, 4, 3).reshape(b * z, -1, s.patch * f)
            return s.head(s.enc(s.proj(p)).mean(1)).reshape(b, z)

    DEEP_MODELS = {"lstm": LSTMNet, "cross_attention_transformer": CrossAttentionNet, "patchtst": PatchTSTNet}
    torch_device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if DEVICE == "mps" and torch.backends.mps.is_available() else "cpu"))

    def fit_deep(hourly, targets, weather, split, model_name, mode, epochs=60, patience=10):
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        parts = sequences(hourly, targets, weather, mode, split)
        (xtr, ytr), (xva, yva), (xte, yte) = parts["train"], parts["validation"], parts["test"]
        # the 24-h lookback of the earliest origins can reach before the first weekly lag: drop those samples
        keep = lambda x, y: ~(np.isnan(x).any(axis=(1, 2, 3)) | np.isnan(y).any(axis=1))
        k = keep(xtr, ytr); xtr, ytr = xtr[k], ytr[k]
        k = keep(xva, yva); xva, yva = xva[k], yva[k]
        assert not np.isnan(xte).any(), "test inputs contain NaN"
        xm, xs = xtr.reshape(-1, xtr.shape[-1]).mean(0), xtr.reshape(-1, xtr.shape[-1]).std(0) + 1e-8
        ym, ys = ytr.mean(0), ytr.std(0) + 1e-8
        nx = lambda a: ((a - xm) / xs).astype(np.float32); ny = lambda a: ((a - ym) / ys).astype(np.float32)
        net = DEEP_MODELS[model_name](xtr.shape[-1], ytr.shape[-1]).to(torch_device)
        opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4); loss_fn = nn.MSELoss()
        loader = DataLoader(TensorDataset(torch.from_numpy(nx(xtr)), torch.from_numpy(ny(ytr))), batch_size=64, shuffle=True)
        vx, vy = torch.from_numpy(nx(xva)).to(torch_device), torch.from_numpy(ny(yva)).to(torch_device)
        best, best_state, wait = float("inf"), copy.deepcopy(net.state_dict()), 0
        for epoch in range(epochs):
            net.train()
            for bx, by in loader:
                opt.zero_grad(); loss = loss_fn(net(bx.to(torch_device)), by.to(torch_device)); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            net.eval()
            with torch.no_grad(): v = loss_fn(net(vx), vy).item()
            if v < best - 1e-7: best, best_state, wait = v, copy.deepcopy(net.state_dict()), 0
            else: wait += 1
            if wait >= patience: break
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad(): pred = net(torch.from_numpy(nx(xte)).to(torch_device)).cpu().numpy() * ys + ym
        print(f"   {model_name} {MODE_NAME[mode]}: stopped after {epoch + 1} epochs")
        return yte, pred

    for model_name in DEEP_MODELS:
        for mode in MODES:
            actual, pred = cached(f"isone_{model_name}_{mode}", lambda: fit_deep(isone, ZONES, ISONE_WEATHER, iso_split, model_name, mode))
            iso_tables.append(record("isone", model_name, mode, actual, pred, ZONES))
        show(pd.concat([t for t in iso_tables if (t["model"] == model_name).any()]), f"\nISO-NE {model_name} (MW):")
else:
    print("RUN_DEEP is False: deep models skipped.")

In [ ]:
# ===================================================================
# STEP 11 — ZERO-SHOT FOUNDATION MODELS (Chronos-Bolt, Chronos-2)
# Context = last 512 h of each zone's load ending at the origin t; forecast t+24.
# Chronos-2 also receives the zone's weather: past values over the context and
# the declared target-time proxy over t+1..t+24. No fine-tuning.
# ===================================================================
FOUNDATION = {"chronos_bolt_small": "amazon/chronos-bolt-small", "chronos_bolt_base": "amazon/chronos-bolt-base", "chronos2": "amazon/chronos-2"}
_pipes = {}
def chronos(name):
    if name not in _pipes:
        import torch
        from chronos import BaseChronosPipeline
        _pipes[name] = BaseChronosPipeline.from_pretrained(FOUNDATION[name], device_map=DEVICE, torch_dtype=torch.float32)
    return _pipes[name]

def chronos_median(name, contexts, horizon, past=None, future=None, batch=256):
    import torch
    pipe, out = chronos(name), []
    for s in range(0, len(contexts), batch):
        if name == "chronos2":
            items = [{"target": contexts[i].astype(np.float32),
                      **({"past_covariates": {k: v.astype(np.float32) for k, v in past[i].items()},
                          "future_covariates": {k: v.astype(np.float32) for k, v in future[i].items()}} if past else {})}
                     for i in range(s, min(s + batch, len(contexts)))]
            q, _ = pipe.predict_quantiles(items, prediction_length=horizon, quantile_levels=[0.5])
            out += [np.asarray(t.float().cpu())[0, :, 0] for t in q]
        else:
            q, _ = pipe.predict_quantiles([torch.tensor(c, dtype=torch.float32) for c in contexts[s:s + batch]], prediction_length=horizon, quantile_levels=[0.5])
            out += list(np.asarray(q.float().cpu())[:, :, 0])
    return np.vstack(out)

def fm_load(hourly, targets, weather, split, name, context=512):
    origins = split["test"]; pos = hourly.index.get_indexer(origins)
    pred = np.zeros((len(origins), len(targets)))
    for j, z in enumerate(targets):
        series = hourly[z].to_numpy(float)
        zw = [c for c in weather if c.startswith(f"{z}_")] or list(weather)
        ctx = [series[max(0, p - context + 1):p + 1] for p in pos]
        past = fut = None
        if name == "chronos2":
            past = [{c: hourly[c].to_numpy()[max(0, p - context + 1):p + 1] for c in zw} for p in pos]
            fut = [{c: hourly[c].to_numpy()[p + 1:p + 1 + H] for c in zw} for p in pos]
        pred[:, j] = chronos_median(name, ctx, H, past, fut)[:, H - 1]
        print(f"   {name}: zone {z} done")
    return hourly[targets].shift(-H).loc[origins].to_numpy(), pred

if RUN_FOUNDATION:
    for name in FOUNDATION:
        actual, pred = cached(f"isone_{name}", lambda: fm_load(isone, ZONES, ISONE_WEATHER, iso_split, name))
        iso_tables.append(record("isone", name, "weather_time_history", actual, pred, ZONES))
    show(pd.concat([t for t in iso_tables if t["model"].isin(FOUNDATION).any()]), "\nISO-NE foundation models, zero-shot (MW):")
else:
    print("RUN_FOUNDATION is False: Chronos skipped.")

In [ ]:
# ===================================================================
# STEP 12 — ISO-NE SIGNIFICANCE: every model vs persistence, per zone
# ===================================================================
def dm_table(dataset, zones, bench):
    base = RESULTS[(dataset, bench, "weather_time_history")]
    rows = []
    for (ds, model, mode), r in RESULTS.items():
        if ds != dataset or model == bench: continue
        for j, z in enumerate(zones):
            em, eb = r["actual"][:, j] - r["pred"][:, j], base["actual"][:, j] - base["pred"][:, j]
            stat, pval = dm_test(em, eb)
            rows.append({"model": model, "set": MODE_NAME.get(mode, mode), "zone": z,
                         "RMSE_ratio": np.sqrt((em ** 2).mean() / (eb ** 2).mean()), "DM_stat": stat, "p_value": pval})
    t = pd.DataFrame(rows)
    return t.groupby(["model", "set"]).agg(ratio_min=("RMSE_ratio", "min"), ratio_max=("RMSE_ratio", "max"),
                                           zones_better=("DM_stat", lambda s: int((s < 0).sum())),
                                           max_p=("p_value", "max")).round(4)

iso_dm = dm_table("isone", ZONES, "persistence")
print("ISO-NE: RMSE ratio vs persistence across zones, zones where the model is better, largest p-value:")
display(iso_dm)

In [ ]:
# ===================================================================
# STEP 13 — ISO-NE LOAD SUMMARY (paper Table 3)
# ===================================================================
iso_all = pd.concat(iso_tables)
iso_summary = iso_all[iso_all.scope == "aggregate"].pivot_table(index="model", columns="set", values="RMSE").round(1)
iso_summary["nRMSE_% (B)"] = iso_all[(iso_all.scope == "aggregate") & (iso_all.set == "B")].set_index("model")["nRMSE_%"].round(2)
display(iso_summary.sort_values("B"))
iso_summary.to_csv(OUT / "table3_isone_load.csv")
iso_summary["B"].sort_values().plot.barh(figsize=(7, 4), color="#2a78d6", title="ISO-NE test RMSE with history (set B), MW")
plt.tight_layout(); plt.show()

In [ ]:
# ===================================================================
# STEP 14 — TETOUAN: same pipeline (benchmarks, trees, deep, Chronos)
# Test window 7 Nov - 29 Dec 2017. Values in dataset units.
# ===================================================================
tet_tables = [benchmarks(tetouan, TET_ZONES, tet_split, "tetouan")]
for model_name in TREE_MODELS:
    for mode in MODES:
        actual, pred = cached(f"tetouan_{model_name}_{mode}", lambda: fit_trees(tet_frames, tet_split, model_name, mode))
        tet_tables.append(record("tetouan", model_name, mode, actual, pred, TET_ZONES))
if RUN_DEEP:
    for model_name in DEEP_MODELS:
        for mode in MODES:
            actual, pred = cached(f"tetouan_{model_name}_{mode}", lambda: fit_deep(tetouan, TET_ZONES, TET_WEATHER, tet_split, model_name, mode))
            tet_tables.append(record("tetouan", model_name, mode, actual, pred, TET_ZONES))
if RUN_FOUNDATION:
    for name in FOUNDATION:
        actual, pred = cached(f"tetouan_{name}", lambda: fm_load(tetouan, TET_ZONES, TET_WEATHER, tet_split, name))
        tet_tables.append(record("tetouan", name, "weather_time_history", actual, pred, TET_ZONES))
tet_all = pd.concat(tet_tables)
tet_summary = tet_all[tet_all.scope == "aggregate"].pivot_table(index="model", columns="set", values="nRMSE_%").round(2)
print("Tetouan single split, nRMSE (%):"); display(tet_summary.sort_values("B"))
print("\nTetouan: every model vs the weekly seasonal naive forecast:")
display(dm_table("tetouan", TET_ZONES, "seasonal_naive"))

In [ ]:
# ===================================================================
# STEP 15 — TETOUAN ROLLING-ORIGIN EVALUATION
# 11 consecutive 28-day test windows from 5 Mar 2017; expanding training set;
# 21-day validation window; 24-h purge between all partitions.
# ===================================================================
def rolling_windows():
    index = tet_frames[MODES[0]][0].index
    out = []
    for start in pd.date_range("2017-03-05", "2017-12-10", freq="28D"):
        h = pd.Timedelta(hours=H); vs = start - pd.Timedelta(days=21)
        out.append((start, {"train": index[index < vs - h], "validation": index[(index >= vs) & (index < start - h)],
                            "test": index[(index >= start) & (index < start + pd.Timedelta(days=28))]}))
    return out

def run_window(start, split):
    rows = []
    test = split["test"]; actual = tetouan[TET_ZONES].shift(-H).loc[test].to_numpy()
    for model, offset in [("persistence", 0), ("seasonal_naive", 168 - H)]:
        pred = tetouan.loc[test - pd.Timedelta(hours=offset), TET_ZONES].to_numpy()
        rows.append(scores(actual, pred, TET_ZONES, model, "weather_time_history"))
    for model_name in TREE_MODELS:
        for mode in MODES:
            a, p = fit_trees(tet_frames, split, model_name, mode)
            rows.append(scores(a, p, TET_ZONES, model_name, mode))
    t = pd.concat(rows); t.insert(0, "window", str(start.date())); return t

roll = pd.concat([cached(f"rolling_{start.date()}", lambda: run_window(start, split)) for start, split in rolling_windows()])
agg = roll[roll.scope == "aggregate"].copy()
agg["rank"] = agg.groupby("window")["RMSE"].rank()
roll_summary = agg.groupby(["model", "set"]).agg(mean_rank=("rank", "mean"), wins=("rank", lambda r: int((r == 1).sum())),
                                                 mean_RMSE=("RMSE", "mean")).sort_values("mean_rank").round(1)
print("Tetouan rolling origin (paper Table 5):"); display(roll_summary)
roll_summary.to_csv(OUT / "table5_tetouan_rolling.csv")

fig, ax = plt.subplots(figsize=(10, 4))
for (model, s) in [("seasonal_naive", "B"), ("persistence", "B"), ("random_forest", "B+"), ("extra_trees", "B+"), ("hist_gradient_boosting", "B+")]:
    d = agg[(agg.model == model) & (agg.set == s)]
    ax.plot(pd.to_datetime(d["window"]), d["RMSE"], marker="o", label=f"{model} ({s})")
for a, b, label in [("2017-05-27", "2017-06-27", "Ramadan / Eid al-Fitr"), ("2017-08-31", "2017-09-03", "Eid al-Adha")]:
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color="#e4e3df", zorder=0)
ax.set_ylabel("RMSE (dataset units)"); ax.set_title("Tetouan: RMSE per 28-day test window"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "figures" / "tetouan_rolling.png", dpi=300); plt.show()

In [ ]:
# ===================================================================
# STEP 16 — DAY-AHEAD PRICE: design matrix (rows = delivery day D)
#  price_only      : prices of D-1, D-2, D-3, D-7 (24 h each) + weekday dummies
#  price_exogenous : + exogenous inputs at their declared lags
#    ISO-NE: weather at D (declared proxy), observed load at D-1 and D-7 only
# ===================================================================
def daily_matrix(series):
    f = series.to_frame("v"); f["day"] = f.index.normalize(); f["hour"] = f.index.hour
    m = f.pivot_table(index="day", columns="hour", values="v", aggfunc="mean")
    return m[m.notna().sum(axis=1) == 24]

def price_design(hourly, exogenous_lags, info):
    price = daily_matrix(hourly["Price"]); days = pd.date_range(price.index.min(), price.index.max(), freq="D")
    price = price.reindex(days); blocks = []
    for lag in (1, 2, 3, 7):
        b = price.shift(lag); b.columns = [f"price_d{lag}_h{h}" for h in range(24)]; blocks.append(b)
    if info == "price_exogenous":
        for col, lags in exogenous_lags.items():
            m = daily_matrix(hourly[col]).reindex(days)
            for lag in lags:
                b = m.shift(lag); b.columns = [f"{col}_d{lag}_h{h}" for h in range(24)]; blocks.append(b)
    dow = pd.get_dummies(pd.Series(days.dayofweek, index=days), prefix="dow").astype(float)
    x = pd.concat([*blocks, dow], axis=1); y = price.copy(); y.columns = [f"h{h}" for h in range(24)]
    ok = x.notna().all(axis=1) & y.notna().all(axis=1)
    return x.loc[ok], y.loc[ok]

ISONE_EXOG = {"Dry_Bulb": (0,), "Dew_Point": (0,), "RT_Demand": (1, 7)}
x_demo, y_demo = price_design(isone_price, ISONE_EXOG, "price_exogenous")
assert not any(c.startswith("RT_Demand_d0") for c in x_demo.columns), "day-D load must never be an input"
print("ISO-NE price design:", x_demo.shape, "| days", x_demo.index.min().date(), "to", x_demo.index.max().date())

In [ ]:
# ===================================================================
# STEP 17 — PRICE MODELS: naive, LEAR, gradient boosting, extra trees, Chronos
# All learners use the asinh variance-stabilising transform (median/MAD).
# Rolling recalibration: LEAR every 7 days, trees every 28 days.
# ===================================================================
from sklearn.linear_model import Lasso, LassoLarsIC

class Asinh:
    def fit(s, v):
        s.med = np.median(v, axis=0); mad = stats.median_abs_deviation(v, axis=0, scale="normal")
        s.mad = np.where(mad > 1e-9, mad, 1.0); return s
    def t(s, v): return np.arcsinh((v - s.med) / s.mad)
    def inv(s, v): return np.sinh(v) * s.mad + s.med

def naive_price(y, day):
    src = day - pd.Timedelta(days=7 if day.dayofweek in (0, 5, 6) else 1)
    return y.loc[src].to_numpy() if src in y.index else np.full(24, np.nan)

def fit_predict_price(model, xtr, ytr, xte, columns):
    sx, sy = Asinh().fit(xtr), Asinh().fit(ytr)
    a, b, yt = sx.t(xtr), sx.t(xte), sy.t(ytr)
    dummies = np.array([c.startswith("dow_") for c in columns]); a[:, dummies], b[:, dummies] = xtr[:, dummies], xte[:, dummies]
    out = np.zeros((len(xte), 24))
    for h in range(24):
        if model == "lear":
            alpha = LassoLarsIC(criterion="aic", max_iter=2500).fit(a, yt[:, h]).alpha_
            est = Lasso(alpha=alpha, max_iter=5000)
        elif model == "hist_gradient_boosting":
            est = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=SEED)
        else:
            est = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.33, n_jobs=-1, random_state=SEED)
        out[:, h] = est.fit(a, yt[:, h]).predict(b)
    return sy.inv(out)

def rolling_price(x, y, test_days, model, calibration_days, every):
    preds = []
    for s in range(0, len(test_days), every):
        block = test_days[s:s + every]; first = block[0]
        mask = x.index < first
        if calibration_days: mask &= x.index >= first - pd.Timedelta(days=calibration_days)
        b = block.intersection(x.index)
        preds.append(pd.DataFrame(fit_predict_price(model, x[mask].to_numpy(float), y[mask].to_numpy(float), x.loc[b].to_numpy(float), list(x.columns)), index=b, columns=y.columns))
    return pd.concat(preds)

def fm_price(hourly, days, name, exogenous_lags, use_exog, context=2048):
    price = hourly["Price"].to_numpy(float); ctx, past, fut = [], [], []
    for d in days:
        e = hourly.index.get_indexer([d])[0]; lo = max(0, e - context)
        ctx.append(price[lo:e])
        if use_exog:
            past.append({c: hourly[c].to_numpy()[lo:e] for c in exogenous_lags})
            fut.append({c: hourly[c].to_numpy()[e:e + 24] for c, lags in exogenous_lags.items() if 0 in lags})
    return chronos_median(name, ctx, 24, past or None, fut or None, batch=64)

def price_experiment(tag, hourly, exogenous_lags, test_start, test_end, calibration_days):
    preds, rows = {}, []
    infos = ["price_only"] + (["price_exogenous"] if exogenous_lags else [])
    for info in infos:
        x, y = price_design(hourly, exogenous_lags, info)
        test_days = y.index[(y.index >= test_start) & (y.index <= test_end)]
        if "naive" not in preds:
            preds["naive"] = pd.DataFrame([naive_price(y, d) for d in test_days], index=test_days, columns=y.columns)
        for model, every in [("lear", 7), ("hist_gradient_boosting", 28), ("extra_trees", 28)]:
            preds[f"{model} | {info}"] = cached(f"price_{tag}_{model}_{info}", lambda: rolling_price(x, y, test_days, model, calibration_days, every))
            print(f"   {tag}: {model} | {info} done")
        if RUN_FOUNDATION:
            for name in FOUNDATION:
                if info == "price_exogenous" and name != "chronos2": continue
                preds[f"{name} | {info}"] = cached(f"price_{tag}_{name}_{info}", lambda: pd.DataFrame(
                    fm_price(hourly, test_days, name, exogenous_lags, info == "price_exogenous"), index=test_days, columns=y.columns))
    common = preds["naive"].dropna().index
    for p in preds.values(): common = common.intersection(p.index)
    truth, naive = y.loc[common].to_numpy(), preds["naive"].loc[common].to_numpy()
    for key, p in preds.items():
        e = truth - p.loc[common].to_numpy()
        rows.append({"model | information set": key, "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
                     "sMAPE_%": 100 * np.mean(2 * np.abs(e) / (np.abs(truth) + np.abs(p.loc[common].to_numpy()) + 1e-9)),
                     "rMAE": np.abs(e).mean() / np.abs(truth - naive).mean()})
    table = pd.DataFrame(rows).sort_values("rMAE").round(3)
    # multivariate DM (Ziel & Weron 2018): p-value that the COLUMN model is better than the ROW model
    keys = list(preds); dm = pd.DataFrame(index=keys, columns=keys, dtype=float)
    for ka in keys:
        for kb in keys:
            if ka != kb:
                la = np.abs(truth - preds[ka].loc[common].to_numpy()).sum(axis=1)
                lb = np.abs(truth - preds[kb].loc[common].to_numpy()).sum(axis=1)
                d = la - lb; dm.loc[ka, kb] = 1 - stats.norm.cdf(d.mean() / np.sqrt(d.var() / len(d)))
    return table, dm.round(3), len(common)

In [ ]:
# ===================================================================
# STEP 18 — ISO-NE DAY-AHEAD PRICE RESULTS (test year 2025, paper Table 4)
# ===================================================================
iso_price_table, iso_price_dm, n_days = price_experiment("isone", isone_price, ISONE_EXOG, "2025-01-01", "2025-12-31", None)
print(f"ISO-NE day-ahead price, {n_days} test days:"); display(iso_price_table)
print("\nMultivariate DM p-values (small = column model significantly better than row model):"); display(iso_price_dm)
iso_price_table.to_csv(OUT / "table4_isone_price.csv"); iso_price_dm.to_csv(OUT / "table4_isone_price_dm.csv")

In [ ]:
# ===================================================================
# STEP 19 — EPF BENCHMARK MARKETS (official files with two day-ahead forecasts)
# Download once from Zenodo; test = last 728 days; 4-year rolling calibration.
# ===================================================================
EPF = {"PJM": ("2016-12-27", "2018-12-24"), "NP": ("2016-12-27", "2018-12-24"),
       "DE": ("2016-01-04", "2017-12-31"), "BE": ("2015-01-04", "2016-12-31"), "FR": ("2015-01-04", "2016-12-31")}
epf_tables = {}
if RUN_EPF:
    import urllib.request
    EPF_DIR.mkdir(parents=True, exist_ok=True)
    for market, (start, end) in EPF.items():
        path = EPF_DIR / f"{market}.csv"
        if not path.exists():
            urllib.request.urlretrieve(f"https://zenodo.org/records/4624805/files/{market}.csv", path)
        raw = pd.read_csv(path, index_col=0, parse_dates=True)
        raw.columns = ["Price"] + [f"Exogenous {i}" for i in range(1, raw.shape[1])]
        lags = {c: (0, 1, 7) for c in raw.columns[1:]}
        table, dm, n = price_experiment(f"epf_{market}", raw.astype(float), lags, start, end, 1456)
        epf_tables[market] = table.set_index("model | information set")["rMAE"]
        print(f"\n{market}: {n} test days"); display(table)
    epf_rmae = pd.DataFrame(epf_tables).round(3)
    print("\nrMAE by market (lower is better; 1.0 = naive):"); display(epf_rmae)
    epf_rmae.to_csv(OUT / "table6_epf_rmae.csv")
else:
    print("RUN_EPF is False: EPF markets skipped.")

In [ ]:
# ===================================================================
# STEP 20 — ALL PAPER TABLES IN ONE PLACE
# ===================================================================
print("Table 3 — ISO-NE load, aggregate RMSE (MW):"); display(iso_summary.sort_values("B"))
print("\nTable 4 — ISO-NE price:"); display(iso_price_table)
print("\nTetouan single split, nRMSE (%):"); display(tet_summary.sort_values("B"))
print("\nTable 5 — Tetouan rolling origin:"); display(roll_summary)
if epf_tables: print("\nEPF markets, rMAE:"); display(epf_rmae)
print("\nAll tables and figures are saved in:", OUT)